In [4]:
%pip install pandas
import pandas as pd

Note: you may need to restart the kernel to use updated packages.


In [5]:

# Loading our attrition data into a pandas dataframe
df = pd.read_csv(
    "../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv"
)

# Visualizing top 5 records
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [ ]:
# Attrition rate by each useful column
# AttritionRate is the share of employees in each group who left.

# Make a new column AttritionFlag that is True for employees who left and False for those who stayed.
df["AttritionFlag"] = df["Attrition"].eq("Yes")

# Compute the overall attrition rate to use as a baseline for comparison.
baseline_rate = df["AttritionFlag"].mean()
print(f"Overall attrition rate: {baseline_rate:.1%}")

# Identify columns that are not useful for attrition analysis.
constant_columns = [
    column for column in df.columns
    if df[column].nunique(dropna=False) <= 1
]
identifier_columns = ["EmployeeNumber"]
exclude_columns = {"Attrition", "AttritionFlag", *constant_columns, *identifier_columns}

results = []
for column in df.columns:
    if column in exclude_columns:
        continue

    grouped_data = df[[column, "AttritionFlag"]].copy()
    group_column = column

    if pd.api.types.is_numeric_dtype(grouped_data[column]):
        group_column = f"{column}Band"
        grouped_data[group_column] = pd.qcut(
            grouped_data[column],
            q=4,
            duplicates="drop"
        )

    grouped_rates = (
        grouped_data
        .groupby(group_column, observed=False)["AttritionFlag"]
        .agg(
            Employees="size",
            Leavers="sum",
            AttritionRate="mean"
        )
        .reset_index()
    )
    grouped_rates.insert(0, "Feature", column)
    grouped_rates["RateVsBaseline"] = grouped_rates["AttritionRate"] - baseline_rate
    results.append(grouped_rates)

attrition_by_group = pd.concat(results, ignore_index=True)
attrition_by_group["AttritionRate"] = attrition_by_group["AttritionRate"].map(
    lambda rate: f"{rate:.1%}"
)
attrition_by_group["RateVsBaseline"] = attrition_by_group["RateVsBaseline"].map(
    lambda rate: f"{rate:+.1%}"
)

# Sort the largest groups first so small, unstable groups do not dominate the view.
attrition_by_group = attrition_by_group.sort_values(
    ["Feature", "Employees"],
    ascending=[True, False]
)

# Pearson correlation with the binary AttritionFlag is a quick numeric screening tool.
numeric_columns = [
    column for column in df.select_dtypes(include="number").columns
    if column not in exclude_columns
]
attrition_correlations = (
    df[numeric_columns]
    .corrwith(df["AttritionFlag"])
    .sort_values(key=lambda values: values.abs(), ascending=False)
    .rename("CorrelationWithAttrition")
    .to_frame()
)

print("\nGrouped attrition rates:")
display(attrition_by_group.head(30))
print("\nNumeric correlations with AttritionFlag:")
display(attrition_correlations)

Overall attrition rate: 16.1%

Grouped attrition rates:


,Feature,AgeBand,Employees,Leavers,AttritionRate,RateVsBaseline,BusinessTravel,DailyRateBand,Department,DistanceFromHomeBand,...,PerformanceRatingBand,RelationshipSatisfactionBand,StockOptionLevelBand,TotalWorkingYearsBand,TrainingTimesLastYearBand,WorkLifeBalanceBand,YearsAtCompanyBand,YearsInCurrentRoleBand,YearsSinceLastPromotionBand,YearsWithCurrManagerBand
1,Age,"(30.0, 36.0]",412,66,16.0%,-0.1%,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,Age,"(17.999, 30.0]",386,100,25.9%,+9.8%,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Age,"(43.0, 60.0]",347,42,12.1%,-4.0%,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Age,"(36.0, 43.0]",325,29,8.9%,-7.2%,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,BusinessTravel,NaN,1043,156,15.0%,-1.2%,Travel_Rarely,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,BusinessTravel,NaN,277,69,24.9%,+8.8%,Travel_Frequently,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,BusinessTravel,NaN,150,12,8.0%,-8.1%,Non-Travel,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,DailyRate,NaN,369,74,20.1%,+3.9%,NaN,"(101.999, 465.0]",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,DailyRate,NaN,367,59,16.1%,-0.0%,NaN,"(465.0, 802.0]",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,DailyRate,NaN,367,56,15.3%,-0.9%,NaN,"(802.0, 1157.0]",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Numeric correlations with AttritionFlag:


,CorrelationWithAttrition
TotalWorkingYears,-0.171063
JobLevel,-0.169105
YearsInCurrentRole,-0.160545
MonthlyIncome,-0.159840
Age,-0.159205
YearsWithCurrManager,-0.156199
StockOptionLevel,-0.137145
YearsAtCompany,-0.134392
JobInvolvement,-0.130016
JobSatisfaction,-0.103481


In [8]:
correlation_data = df[numeric_columns].copy()
correlation_data["AttritionFlag"] = df["AttritionFlag"].astype(int)

correlation_data.corr()


,Age,DailyRate,DistanceFromHome,Education,EnvironmentSatisfaction,HourlyRate,JobInvolvement,JobLevel,JobSatisfaction,MonthlyIncome,...,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,AttritionFlag
Age,1.000000,0.010661,-0.001686,0.208034,0.010146,0.024287,0.029820,0.509604,-0.004892,0.497855,...,0.053535,0.037510,0.680381,-0.019621,-0.021490,0.311309,0.212901,0.216513,0.202089,-0.159205
DailyRate,0.010661,1.000000,-0.004985,-0.016806,0.018355,0.023381,0.046135,0.002966,0.030571,0.007707,...,0.007846,0.042143,0.014515,0.002453,-0.037848,-0.034055,0.009932,-0.033229,-0.026363,-0.056652
DistanceFromHome,-0.001686,-0.004985,1.000000,0.021042,-0.016075,0.031131,0.008783,0.005303,-0.003669,-0.017014,...,0.006557,0.044872,0.004628,-0.036942,-0.026556,0.009508,0.018845,0.010029,0.014406,0.077924
Education,0.208034,-0.016806,0.021042,1.000000,-0.027128,0.016775,0.042438,0.101589,-0.011296,0.094961,...,-0.009118,0.018422,0.148280,-0.025100,0.009819,0.069114,0.060236,0.054254,0.069065,-0.031373
EnvironmentSatisfaction,0.010146,0.018355,-0.016075,-0.027128,1.000000,-0.049857,-0.008278,0.001212,-0.006784,-0.006259,...,0.007665,0.003432,-0.002693,-0.019359,0.027627,0.001458,0.018007,0.016194,-0.004999,-0.103369
HourlyRate,0.024287,0.023381,0.031131,0.016775,-0.049857,1.000000,0.042861,-0.027853,-0.071335,-0.015794,...,0.001330,0.050263,-0.002334,-0.008548,-0.004607,-0.019582,-0.024106,-0.026716,-0.020123,-0.006846
JobInvolvement,0.029820,0.046135,0.008783,0.042438,-0.008278,0.042861,1.000000,-0.012630,-0.021476,-0.015271,...,0.034297,0.021523,-0.005533,-0.015338,-0.014617,-0.021355,0.008717,-0.024184,0.025976,-0.130016
JobLevel,0.509604,0.002966,0.005303,0.101589,0.001212,-0.027853,-0.012630,1.000000,-0.001944,0.950300,...,0.021642,0.013984,0.782208,-0.018191,0.037818,0.534739,0.389447,0.353885,0.375281,-0.169105
JobSatisfaction,-0.004892,0.030571,-0.003669,-0.011296,-0.006784,-0.071335,-0.021476,-0.001944,1.000000,-0.007157,...,-0.012454,0.010690,-0.020185,-0.005779,-0.019459,-0.003803,-0.002305,-0.018214,-0.027656,-0.103481
MonthlyIncome,0.497855,0.007707,-0.017014,0.094961,-0.006259,-0.015794,-0.015271,0.950300,-0.007157,1.000000,...,0.025873,0.005408,0.772893,-0.021736,0.030683,0.514285,0.363818,0.344978,0.344079,-0.159840
